# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 5 – Root Extraction Using the Quranic Arabic Corpus (QAC)

## Objective

This notebook implements the root extraction stage of the proposed retrieval framework.

The automatically selected query terms produced in Notebook 4 are mapped to their corresponding Qur'anic roots using the processed Quranic Arabic Corpus (QAC).

Rather than relying on rule-based stemming algorithms, this study utilizes the manually annotated morphological information available in the QAC. This resource provides linguistically validated roots that preserve the lexical characteristics of Qur'anic Arabic.

The extracted roots constitute the bridge between the user query and the semantic knowledge contained in *Maqāyīs al-Lughah*. The resulting roots will therefore serve as the input to the lexical semantic expansion stage implemented in Notebook 6.

In [1]:
# ============================================================
# Standard Library
# ============================================================

from pathlib import Path

# ============================================================
# Third-Party Libraries
# ============================================================

import pandas as pd

In [2]:
# ============================================================
# Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 120)

print("✅ Display settings configured successfully.")

✅ Display settings configured successfully.


In [3]:
# ============================================================
# Define Project Paths
# ============================================================

import os
from pathlib import Path


def find_workspace_root(start=None):
    """
    Locate the current thesis workspace without hard-coding a
    user-specific absolute path.
    """

    configured = os.environ.get("THESIS_WORKSPACE_ROOT")

    if configured:
        configured_path = Path(configured).expanduser().resolve()

        if (configured_path / "02_Methodology_Evidence").is_dir():
            return configured_path

        raise FileNotFoundError(
            "THESIS_WORKSPACE_ROOT is set, but it does not contain "
            "'02_Methodology_Evidence'."
        )

    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "02_Methodology_Evidence").is_dir():
            return candidate

        expected_child = (
            candidate /
            "Master Thesis - Quranic Passage Retrieval - Writing Workspace"
        )

        if (expected_child / "02_Methodology_Evidence").is_dir():
            return expected_child

    raise FileNotFoundError(
        "Could not locate the thesis workspace. Start Jupyter from the "
        "workspace (or one of its subfolders), or set THESIS_WORKSPACE_ROOT."
    )


PROJECT_ROOT = find_workspace_root()

METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"

QAC_ROOT_EXTRACTION_DIR = (
    METHODOLOGY_DIR /
    "QAC and Root Extraction"
)

QAC_DIR = (
    QAC_ROOT_EXTRACTION_DIR /
    "QAC Data" /
    "QAC_Processed"
)

CORRECTED_OUTPUT_DIR = (
    QAC_ROOT_EXTRACTION_DIR /
    "Corrected_Outputs" /
    "Point_2_Query_Root_Provenance"
)

CORRECTED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Input Files
# ------------------------------------------------------------

# Corrected output of Notebook 4
QUERY_FILE = (
    CORRECTED_OUTPUT_DIR /
    "QuranQA_Preprocessed_Queries_Corrected.csv"
)

# Preserved QAC lookup resource generated by Notebook 5A
WORD_LOOKUP_FILE = QAC_DIR / "QAC_Word_Lookup.csv"

# Root-level dictionary (supporting metadata / validation)
ROOT_DICTIONARY_FILE = QAC_DIR / "QAC_Root_Dictionary.csv"

print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print(f"Project Root          : {PROJECT_ROOT}")
print(f"QAC Directory         : {QAC_DIR}")
print(f"Corrected Output Dir  : {CORRECTED_OUTPUT_DIR}")

print("\nInput Files")
print(f"Query File            : {QUERY_FILE}")
print(f"Word Lookup File      : {WORD_LOOKUP_FILE}")
print(f"Root Dictionary       : {ROOT_DICTIONARY_FILE}")

print("\nValidation")
print(f"Query File exists     : {QUERY_FILE.exists()}")
print(f"Word Lookup exists    : {WORD_LOOKUP_FILE.exists()}")
print(f"Root Dictionary exists: {ROOT_DICTIONARY_FILE.exists()}")


PROJECT PATHS
Project Root          : ${THESIS_WORKSPACE}
QAC Directory         : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\QAC Data\QAC_Processed
Corrected Output Dir  : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance

Input Files
Query File            : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance\QuranQA_Preprocessed_Queries_Corrected.csv
Word Lookup File      : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\QAC Data\QAC_Processed\QAC_Word_Lookup.csv
Root Dictionary       : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\QAC Data\QAC_Processed\QAC_Root_Dictionary.csv

Validation
Query File exists     : True
Word Lookup exists    : True
Root Dictionary exists: True


## Loading the Required Resources

This notebook integrates three processed resources produced during the previous stages of the framework:

- **Preprocessed QuranQA Queries** generated in Notebook 4.
- **QAC Word Lookup Table** constructed in Notebook 5A.
- **QAC Root Dictionary** containing root-level linguistic information.

These resources collectively provide the lexical and semantic information required for QAC-based root retrieval. The preprocessed queries supply the normalized query terms, the word lookup table enables lexical matching, and the root dictionary provides validated root-level metadata for the retrieved roots.

In [4]:
# ============================================================
# Load the Required Resources
# ============================================================

# Load corrected preprocessed QuranQA queries
queries_df = pd.read_csv(QUERY_FILE)

# Load the preserved QAC word lookup table
word_lookup_df = pd.read_csv(WORD_LOOKUP_FILE)

# Load the QAC root dictionary
root_dictionary_df = pd.read_csv(ROOT_DICTIONARY_FILE)

print("=" * 70)
print("RESOURCES LOADED")
print("=" * 70)

summary_df = pd.DataFrame({
    "Resource": [
        "Corrected Preprocessed Queries",
        "QAC Word Lookup Table",
        "QAC Root Dictionary"
    ],
    "Records": [
        len(queries_df),
        len(word_lookup_df),
        len(root_dictionary_df)
    ]
})

display(summary_df)

print("\nPreprocessed Query Columns")
print(list(queries_df.columns))

print("\nWord Lookup Columns")
print(list(word_lookup_df.columns))

print("\nRoot Dictionary Columns")
print(list(root_dictionary_df.columns))

required_query_columns = {
    "Dataset",
    "Question_ID",
    "Question",
    "Query_Terms",
    "Query_Term_Provenance"
}

missing_query_columns = required_query_columns - set(queries_df.columns)

if missing_query_columns:
    raise ValueError(
        "Corrected Notebook-4 output is missing required columns: "
        + ", ".join(sorted(missing_query_columns))
    )


RESOURCES LOADED


,Resource,Records
0,Corrected Preprocessed Queries,251
1,QAC Word Lookup Table,11781
2,QAC Root Dictionary,1642



Preprocessed Query Columns
['Dataset', 'Question_ID', 'Question', 'Normalized_Question', 'Original_Clean_Question', 'Clean_Question', 'Original_Tokens', 'Tokens', 'Content_Tokens', 'Query_Terms', 'Query_Term_Provenance']

Word Lookup Columns
['Word_AR', 'Lemma_AR', 'Root_AR', 'POS', 'Word_BW', 'Lemma_BW', 'Root_BW', 'Surah', 'Ayah', 'Verse_ID', 'Word_Normalized', 'Lemma_Normalized']

Root Dictionary Columns
['Root_BW', 'Root_AR', 'Root_Frequency', 'Number_of_Verses', 'Number_of_Words', 'Example_Word', 'Example_Lemma', 'POS_Examples', 'Ibn_Faris_Core', 'English_Gloss', 'Semantic_Domain', 'Validated', 'Notes']


## Building Fast Lookup Dictionaries

The QAC Word Lookup Table and the QAC Root Dictionary are converted into dictionary-based lookup structures to enable efficient retrieval during root extraction.

Instead of repeatedly searching the tabular datasets, dictionary-based indexing provides direct access to lexical entries using normalized Arabic words, normalized lemmas, and validated roots. This preprocessing step improves the efficiency of the retrieval process while preserving the original linguistic annotations.

In [5]:
# ============================================================
# Build Lookup Structures
# ============================================================

# The original implementation collapsed each normalized QAC form to the
# first row. That is unsafe when normalization causes several QAC entries
# with different roots to share the same normalized spelling.
#
# The corrected implementation preserves every candidate row and resolves
# a QAC root only when the candidate set contains one unambiguous valid root.

word_groups = {
    key: group.copy()
    for key, group in word_lookup_df.groupby(
        "Word_Normalized",
        sort=False,
        dropna=False
    )
    if pd.notna(key) and str(key).strip()
}

lemma_groups = {
    key: group.copy()
    for key, group in word_lookup_df.groupby(
        "Lemma_Normalized",
        sort=False,
        dropna=False
    )
    if pd.notna(key) and str(key).strip()
}

# Root lookup is retained for supporting metadata / inspection.
root_lookup = (
    root_dictionary_df
    .drop_duplicates(subset="Root_AR")
    .set_index("Root_AR")
    .to_dict(orient="index")
)

print("=" * 70)
print("LOOKUP STRUCTURES CREATED")
print("=" * 70)

print(f"Normalized Word Groups  : {len(word_groups):,}")
print(f"Normalized Lemma Groups : {len(lemma_groups):,}")
print(f"Root Lookup Entries     : {len(root_lookup):,}")

# ------------------------------------------------------------
# Collision diagnostic
# ------------------------------------------------------------

def valid_root_set(series):
    return sorted({
        str(value).strip()
        for value in series
        if pd.notna(value) and str(value).strip()
    })


conflicting_word_groups = []

for key, group in word_groups.items():

    roots = valid_root_set(group["Root_AR"])

    if len(roots) > 1:
        conflicting_word_groups.append({
            "Normalized_Form": key,
            "Candidate_Count": int(len(group)),
            "Candidate_Roots": roots
        })

conflicting_word_groups_df = pd.DataFrame(conflicting_word_groups)

print(
    "Normalized word groups with conflicting roots : "
    f"{len(conflicting_word_groups_df):,}"
)

if len(conflicting_word_groups_df):
    display(conflicting_word_groups_df.head(20))


LOOKUP STRUCTURES CREATED
Normalized Word Groups  : 7,319
Normalized Lemma Groups : 4,229
Root Lookup Entries     : 1,642


Normalized word groups with conflicting roots : 97


,Normalized_Form,Candidate_Count,Candidate_Roots
0,سم,3,"[سمم, سمو]"
1,زاد,3,"[زود, زيد]"
2,كل,10,"[اكل, كلل, كيل]"
3,ينقض,2,"[قضض, نقض]"
4,امر,12,"[امر, مرر]"
5,اقل,4,"[قلل, قول]"
6,قل,5,"[قلل, قول]"
7,ابي,5,"[ابو, ابي]"
8,هدا,2,"[هدد, هدي]"
9,هاد,3,"[هدي, هود]"


## QAC-Based Root Retrieval

The preprocessed query terms are matched against the normalized QAC Word Lookup Table to retrieve their corresponding Qur'anic roots.

For each query term, the retrieval process follows a hierarchical strategy:

1. Search for an exact match using the normalized Arabic word.
2. If no match is found, search using the normalized lemma.
3. If a root is retrieved, enrich it with the corresponding linguistic information from the QAC Root Dictionary.
4. If no match is found, the term is recorded as unmatched.

This retrieval strategy combines lexical matching with linguistically validated root information while preserving the original QAC annotations.

In [6]:
# ============================================================
# QAC-Based Root Retrieval
# ============================================================

import ast
import json


def json_scalar(value):
    """
    Convert pandas/numpy missing values and scalar objects to
    standards-compliant JSON-friendly Python values.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return value


def clean_list(values):
    """
    Return a deterministic list of non-empty JSON-safe values.
    """

    cleaned = []

    for value in values:
        value = json_scalar(value)

        if value is None:
            continue

        value = str(value).strip()

        if value:
            cleaned.append(value)

    return sorted(set(cleaned))


def deterministic_candidate(group, required_root=None):
    """
    Select a stable representative row for provenance only.

    A representative row is selected only after the root decision has
    already been made from the complete candidate set. This function
    never decides which root should win.
    """

    candidate_group = group.copy()

    if required_root is not None:
        candidate_group = candidate_group[
            candidate_group["Root_AR"].apply(
                lambda x: (
                    str(x).strip()
                    if pd.notna(x)
                    else None
                )
            ) == required_root
        ].copy()

    sort_columns = [
        column
        for column in [
            "Verse_ID",
            "Surah",
            "Ayah",
            "Word_AR",
            "Lemma_AR",
            "POS"
        ]
        if column in candidate_group.columns
    ]

    if sort_columns:
        candidate_group = candidate_group.sort_values(
            sort_columns,
            kind="stable",
            na_position="last"
        )

    if len(candidate_group) == 0:
        return None

    return candidate_group.iloc[0]


def get_candidate_group(term):
    """
    Apply the existing hierarchical QAC lookup order while retaining
    the complete candidate group from the first successful lookup stage.

    Strategy:
    1. Exact normalized word lookup.
    2. If not found and the term begins with ال, remove the article and:
       a. search normalized word,
       b. search normalized lemma.
    3. Exact normalized lemma lookup.
    """

    if term in word_groups:
        return word_groups[term], "Word"

    if term.startswith("ال") and len(term) > 2:

        stripped_term = term[2:]

        if stripped_term in word_groups:
            return word_groups[stripped_term], "Word (without ال)"

        if stripped_term in lemma_groups:
            return lemma_groups[stripped_term], "Lemma (without ال)"

    if term in lemma_groups:
        return lemma_groups[term], "Lemma"

    return None, "-"


def resolve_qac_candidates(term, original_term):
    """
    Resolve a QAC candidate set conservatively.

    Resolution rules:
    - no candidate group -> NO_MATCH;
    - lexical candidate group but no valid root -> LEXICAL_MATCH_NO_ROOT;
    - one unique valid root -> RESOLVED;
    - conflicting valid roots -> AMBIGUOUS_ROOT.

    Conflicting roots are not resolved by taking the first row.
    """

    candidates, lookup_type = get_candidate_group(term)

    if candidates is None or len(candidates) == 0:
        return {
            "Query_Term": term,
            "Original_Term": original_term,
            "Normalized_Term": term,
            "Lookup_Type": "-",
            "Matched": False,
            "Lexical_Matched": False,
            "Root_Resolved": False,
            "QAC_Status": "NO_MATCH",
            "Word_AR": None,
            "Lemma_AR": None,
            "Root_AR": None,
            "POS": None,
            "QAC_Candidate_Count": 0,
            "QAC_Candidate_Words": [],
            "QAC_Candidate_Lemmas": [],
            "QAC_Candidate_Roots": [],
            "QAC_Candidate_POS": []
        }

    candidate_roots = clean_list(candidates["Root_AR"])
    candidate_words = clean_list(candidates["Word_AR"])
    candidate_lemmas = clean_list(candidates["Lemma_AR"])
    candidate_pos = clean_list(candidates["POS"])

    lexical_matched = True

    if len(candidate_roots) == 0:
        status = "LEXICAL_MATCH_NO_ROOT"
        root_resolved = False
        selected = deterministic_candidate(candidates)
        selected_root = None

    elif len(candidate_roots) == 1:
        status = "RESOLVED"
        root_resolved = True
        selected_root = candidate_roots[0]
        selected = deterministic_candidate(
            candidates,
            required_root=selected_root
        )

    else:
        status = "AMBIGUOUS_ROOT"
        root_resolved = False
        selected = None
        selected_root = None

    return {
        "Query_Term": term,
        "Original_Term": original_term,
        "Normalized_Term": term,
        "Lookup_Type": lookup_type,

        # Matched preserves the legacy lexical-match meaning in Notebook 5.
        # Root_Resolved is the authoritative root-success flag.
        "Matched": lexical_matched,
        "Lexical_Matched": lexical_matched,
        "Root_Resolved": root_resolved,
        "QAC_Status": status,

        "Word_AR": (
            json_scalar(selected["Word_AR"])
            if selected is not None
            else None
        ),
        "Lemma_AR": (
            json_scalar(selected["Lemma_AR"])
            if selected is not None
            else None
        ),
        "Root_AR": selected_root,
        "POS": (
            json_scalar(selected["POS"])
            if selected is not None
            else None
        ),

        "QAC_Candidate_Count": int(len(candidates)),
        "QAC_Candidate_Words": candidate_words,
        "QAC_Candidate_Lemmas": candidate_lemmas,
        "QAC_Candidate_Roots": candidate_roots,
        "QAC_Candidate_POS": candidate_pos
    }


# ============================================================
# Root Retrieval
# ============================================================

retrieval_results = []

for _, row in queries_df.iterrows():

    question_results = []

    query_terms = ast.literal_eval(row["Query_Terms"])

    provenance_records = json.loads(
        row["Query_Term_Provenance"]
    )

    provenance_map = {
        item["Normalized_Term"]: item["Original_Term"]
        for item in provenance_records
    }

    for term in query_terms:

        original_term = provenance_map.get(term, term)

        question_results.append(
            resolve_qac_candidates(
                term=term,
                original_term=original_term
            )
        )

    retrieval_results.append({

        "Dataset": row["Dataset"],
        "Question_ID": row["Question_ID"],
        "Question": row["Question"],
        "Query_Terms": query_terms,
        "Query_Term_Provenance": provenance_records,
        "Lexical_Matches": question_results

    })

print("=" * 70)
print("QAC ROOT RETRIEVAL COMPLETED")
print("=" * 70)

print(f"Questions Processed : {len(retrieval_results):,}")

print("\nSample Retrieval Result\n")

sample = retrieval_results[0]

print(f"Question ID : {sample['Question_ID']}")
print(f"Question    : {sample['Question']}")

display(pd.DataFrame(sample["Lexical_Matches"]))


QAC ROOT RETRIEVAL COMPLETED
Questions Processed : 251

Sample Retrieval Result

Question ID : 101
Question    : من هم قوم شعيب؟


,Query_Term,Original_Term,Normalized_Term,Lookup_Type,Matched,Lexical_Matched,Root_Resolved,QAC_Status,Word_AR,Lemma_AR,Root_AR,POS,QAC_Candidate_Count,QAC_Candidate_Words,QAC_Candidate_Lemmas,QAC_Candidate_Roots,QAC_Candidate_POS
0,قوم,قوم,قوم,Word,True,True,True,RESOLVED,قَوْمٍ,قَوْم,قوم,N,6,"[قَوْمٌ, قَوْمٍ, قَوْمَ, قَوْمُ, قَوْمِ, قُومُ]","[قَامَ, قَوْم]",[قوم],"[N, V]"
1,شعيب,شعيب,شعيب,Word,True,True,False,LEXICAL_MATCH_NO_ROOT,شُعَيْبٌ,شُعَيْب,NaN,PN,2,"[شُعَيْبٌ, شُعَيْبُ]",[شُعَيْب],[],[PN]


## Evaluation of the QAC-Based Root Retrieval

The performance of the retrieval process is evaluated by measuring the number of successfully matched query terms, validated roots, proper nouns without root annotations, and unmatched terms.

This evaluation provides a quantitative assessment of the effectiveness of the QAC Word Lookup Table and the lexical retrieval strategy before the retrieved roots are exported for subsequent processing.

In [7]:
# ============================================================
# Evaluate QAC-Based Root Retrieval
# ============================================================

total_terms = 0
lexical_matches = 0
resolved_roots = 0
lexical_no_root = 0
ambiguous_roots = 0
no_matches = 0

for question in retrieval_results:

    for match in question["Lexical_Matches"]:

        total_terms += 1

        if match["Lexical_Matched"]:
            lexical_matches += 1

        if match["Root_Resolved"]:
            resolved_roots += 1

        if match["QAC_Status"] == "LEXICAL_MATCH_NO_ROOT":
            lexical_no_root += 1

        elif match["QAC_Status"] == "AMBIGUOUS_ROOT":
            ambiguous_roots += 1

        elif match["QAC_Status"] == "NO_MATCH":
            no_matches += 1

evaluation_df = pd.DataFrame({

    "Metric": [
        "Total Query Terms",
        "QAC Lexical Matches",
        "QAC Terms with Resolved Roots",
        "Lexical Matches without Roots",
        "Ambiguous QAC Root Groups",
        "No QAC Match",
        "Lexical Match Rate (%)",
        "QAC Root Resolution Rate (%)"
    ],

    "Value": [
        total_terms,
        lexical_matches,
        resolved_roots,
        lexical_no_root,
        ambiguous_roots,
        no_matches,
        round((lexical_matches / total_terms) * 100, 2),
        round((resolved_roots / total_terms) * 100, 2)
    ]
})

print("=" * 70)
print("QAC ROOT RETRIEVAL EVALUATION")
print("=" * 70)

display(evaluation_df)

# Strict validity check: a QAC-resolved record must contain one usable root.
invalid_resolved = []

for question in retrieval_results:
    for match in question["Lexical_Matches"]:
        if match["Root_Resolved"] and not match["Root_AR"]:
            invalid_resolved.append(
                (question["Question_ID"], match["Query_Term"])
            )

if invalid_resolved:
    raise ValueError(
        "QAC root-resolution validation failed: "
        f"{len(invalid_resolved)} resolved records have no root."
    )

print("\nRoot-resolution validation passed.")


QAC ROOT RETRIEVAL EVALUATION


,Metric,Value
0,Total Query Terms,1342.00
1,QAC Lexical Matches,599.00
2,QAC Terms with Resolved Roots,525.00
3,Lexical Matches without Roots,50.00
4,Ambiguous QAC Root Groups,24.00
5,No QAC Match,743.00
6,Lexical Match Rate (%),44.63
7,QAC Root Resolution Rate (%),39.12



Root-resolution validation passed.


## Analysis of Unmatched Query Terms

To better understand the performance of the QAC-based retrieval process, the unmatched query terms are analyzed.

This analysis identifies the most frequent terms that could not be matched against the QAC Word Lookup Table. Examining these terms helps distinguish between genuine retrieval failures and expected vocabulary differences, such as modern question words or explanatory expressions that do not occur as lexical items in the Qur'anic Arabic Corpus.

The results provide valuable insights into the strengths and limitations of the lexical retrieval stage.

In [8]:
# ============================================================
# Analyze Terms Requiring Fallback
# ============================================================

# Notebook 6 must process every term for which QAC did not provide an
# acceptable root, including lexical matches without roots and ambiguous
# normalized forms. This is broader and more precise than "no lexical match".

fallback_terms = []

for question in retrieval_results:

    for match in question["Lexical_Matches"]:

        if not match["Root_Resolved"]:
            fallback_terms.append({
                "Dataset": question["Dataset"],
                "Question_ID": question["Question_ID"],
                "Question": question["Question"],
                "Original_Term": match["Original_Term"],
                "Normalized_Term": match["Normalized_Term"],
                "QAC_Status": match["QAC_Status"],
                "QAC_Candidate_Roots": match["QAC_Candidate_Roots"]
            })

fallback_df = pd.DataFrame(fallback_terms)

print("=" * 70)
print("QUERY TERMS REQUIRING CAMeL FALLBACK")
print("=" * 70)

print(f"Total Fallback Terms : {len(fallback_df):,}")

if len(fallback_df):

    print(
        f"Unique Normalized Terms : "
        f"{fallback_df['Normalized_Term'].nunique():,}"
    )

    print("\nFallback Status Counts")
    display(
        fallback_df["QAC_Status"]
        .value_counts()
        .rename_axis("QAC_Status")
        .reset_index(name="Count")
    )

    print("\nRepresentative Records")
    display(fallback_df.head(30))


QUERY TERMS REQUIRING CAMeL FALLBACK
Total Fallback Terms : 817
Unique Normalized Terms : 517

Fallback Status Counts


,QAC_Status,Count
0,NO_MATCH,743
1,LEXICAL_MATCH_NO_ROOT,50
2,AMBIGUOUS_ROOT,24



Representative Records


,Dataset,Question_ID,Question,Original_Term,Normalized_Term,QAC_Status,QAC_Candidate_Roots
0,Train,101,من هم قوم شعيب؟,شعيب,شعيب,LEXICAL_MATCH_NO_ROOT,[]
1,Train,102,من هم قوم موسى؟,موسى,موسي,LEXICAL_MATCH_NO_ROOT,[]
2,Train,105,من هو النبي المعروف بالصبر؟,بالصبر,بالصبر,NO_MATCH,[]
3,Train,106,من كفل السيدة مريم؟,السيدة,السيدة,NO_MATCH,[]
4,Train,106,من كفل السيدة مريم؟,مريم,مريم,LEXICAL_MATCH_NO_ROOT,[]
5,Train,107,ما معنى الحطمة؟,معنى,معني,NO_MATCH,[]
6,Train,108,من هو اخو سيدنا موسى؟,سيدنا,سيدنا,NO_MATCH,[]
7,Train,108,من هو اخو سيدنا موسى؟,موسى,موسي,LEXICAL_MATCH_NO_ROOT,[]
8,Train,109,ما معنى القارعة؟,معنى,معني,NO_MATCH,[]
9,Train,110,ما معنى الجاثية؟,معنى,معني,NO_MATCH,[]


## Exporting the QAC Root Retrieval Results

The retrieved lexical matches and QAC-derived roots are exported for subsequent processing.

The exported dataset preserves the original query information together with the lexical retrieval results obtained from the Quranic Arabic Corpus (QAC). This resource serves as the input to the next stage of the framework, where unmatched terms will be processed using additional Arabic lexical resources.

In [9]:
# ============================================================
# Export QAC Root Retrieval Results
# ============================================================

import json

export_df = pd.DataFrame({
    "Dataset": [r["Dataset"] for r in retrieval_results],
    "Question_ID": [r["Question_ID"] for r in retrieval_results],
    "Question": [r["Question"] for r in retrieval_results],
    "Query_Terms": [
        json.dumps(
            r["Query_Terms"],
            ensure_ascii=False,
            allow_nan=False
        )
        for r in retrieval_results
    ],
    "Query_Term_Provenance": [
        json.dumps(
            r["Query_Term_Provenance"],
            ensure_ascii=False,
            allow_nan=False
        )
        for r in retrieval_results
    ],
    "Lexical_Matches": [
        json.dumps(
            r["Lexical_Matches"],
            ensure_ascii=False,
            allow_nan=False
        )
        for r in retrieval_results
    ]
})

OUTPUT_FILE = (
    CORRECTED_OUTPUT_DIR /
    "QuranQA_QAC_Root_Retrieval_Corrected.csv"
)

export_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("QAC ROOT RETRIEVAL RESULTS SAVED")
print("=" * 70)

print(f"Output File : {OUTPUT_FILE.name}")
print(f"Location    : {OUTPUT_FILE.parent}")

print(f"\nQuestions Exported : {len(export_df):,}")

print("\nColumns")

for column in export_df.columns:
    print(f"- {column}")

print(
    "\nPreservation rule: the historical "
    "'QuranQA_QAC_Root_Retrieval.csv' file is not overwritten."
)


QAC ROOT RETRIEVAL RESULTS SAVED
Output File : QuranQA_QAC_Root_Retrieval_Corrected.csv
Location    : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance

Questions Exported : 251

Columns
- Dataset
- Question_ID
- Question
- Query_Terms
- Query_Term_Provenance
- Lexical_Matches

Preservation rule: the historical 'QuranQA_QAC_Root_Retrieval.csv' file is not overwritten.


# Summary

This notebook implemented the QAC-based lexical retrieval stage of the proposed framework.

The workflow included:

- loading the corrected preprocessed QuranQA queries,
- loading the QAC Word Lookup Table and the QAC Root Dictionary,
- retaining the complete QAC candidate group for each normalized lookup form,
- retrieving lexical matches and accepting a QAC root only when the candidate evidence supports one unambiguous valid root,
- preserving lexical matches without roots and ambiguous QAC root groups as explicit fallback cases,
- evaluating lexical matching separately from root resolution, and
- exporting complete query-term and QAC-candidate provenance for the CAMeL fallback stage.

The correction does not introduce a new disambiguation model. It removes the previous first-row dependency: when normalization produces conflicting QAC roots for the same lookup form, the root remains unresolved at the QAC stage and is passed to the existing CAMeL fallback. If neither route later provides an acceptable root, the term remains unresolved.

The exported retrieval results provide the authoritative QAC evidence used by **Notebook 6**, which performs the CAMeL fallback and constructs the hybrid query-root representation.
